In [0]:
raw_path = "/Volumes/ecommerce/default/ecommerce_raw"
display(dbutils.fs.ls(raw_path))

path,name,size,modificationTime
dbfs:/Volumes/ecommerce/default/ecommerce_raw/bronze/,bronze/,0,1790577767093
dbfs:/Volumes/ecommerce/default/ecommerce_raw/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1790416191000
dbfs:/Volumes/ecommerce/default/ecommerce_raw/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1790416204000
dbfs:/Volumes/ecommerce/default/ecommerce_raw/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1790416194000
dbfs:/Volumes/ecommerce/default/ecommerce_raw/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1790416190000
dbfs:/Volumes/ecommerce/default/ecommerce_raw/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1790416193000
dbfs:/Volumes/ecommerce/default/ecommerce_raw/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1790416194000
dbfs:/Volumes/ecommerce/default/ecommerce_raw/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1790416187000
dbfs:/Volumes/ecommerce/default/ecommerce_raw/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1790416186000
dbfs:/Volumes/ecommerce/default/ecommerce_raw/product_category_name_translation.csv,product_category_name_translation.csv,2613,1790416186000


In [0]:
#Reading CSV Files using PySpark
from pyspark.sql import functions as F

files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

dataframes = {}

for table_name, file_name in files.items():
    file_path = f"{raw_path}/{file_name}"
    
    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(file_path)
    )

    dataframes[table_name] = df

    print(f"{table_name}: {df.count()} records")

customers: 99441 records
geolocation: 1000163 records
orders: 99441 records
order_items: 112650 records
order_payments: 103886 records
products: 32951 records
sellers: 3095 records
category_translation: 71 records


In [0]:
#Reading order_reviews separately because of multiline comments
order_reviews_path = f"{raw_path}/olist_order_reviews_dataset.csv"

order_reviews = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .load(order_reviews_path)
)

dataframes["order_reviews"] = order_reviews

print("Total Records:", order_reviews.count())

Total Records: 99224


In [0]:
for table_name, df in dataframes.items():
    print(f"\nTable: {table_name}")
    print(f"Columns: {df.columns}")


Table: customers
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Table: geolocation
Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Table: orders
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Table: order_items
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Table: order_payments
Columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Table: products
Columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Table: sellers
Columns: ['seller_i

In [0]:
#BRONZE DELTA TABBLE
bronze_path = "/Volumes/ecommerce/default/ecommerce_raw/bronze"

for table_name, df in dataframes.items():
    target_path = f"{bronze_path}/{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
    )

    print(f"Created Bronze table: {table_name}")

Created Bronze table: customers
Created Bronze table: geolocation
Created Bronze table: orders
Created Bronze table: order_items
Created Bronze table: order_payments
Created Bronze table: products
Created Bronze table: sellers
Created Bronze table: category_translation
Created Bronze table: order_reviews


In [0]:
for table_name in files:
    table_path = f"{bronze_path}/{table_name}"
    
    df= spark.read.format("delta").load(table_path)
    
    print(f"\nTable: {table_name}")
    print(f"Rows: {df.count()}")
    df.printSchema()


Table: customers
Rows: 99441
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)


Table: geolocation
Rows: 1000163
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)


Table: orders
Rows: 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: